# GenCultura VLM Server
Runs **Qwen2.5-VL-7B-Instruct** on an A100 GPU via vLLM and exposes it as an OpenAI-compatible API.

**Steps:**
1. Runtime → Change runtime type → **A100 GPU**
2. Get a free ngrok token at https://dashboard.ngrok.com/auth
3. Run all cells top to bottom
4. Copy the `.env` block printed by Cell 5 into your backend `.env`
5. Restart the backend

In [ ]:
# Cell 1 — Install (takes ~3 minutes, run once)
!pip install vllm pyngrok -q

In [ ]:
# Cell 2 — Configuration
NGROK_TOKEN = "paste-your-ngrok-token-here"  # https://dashboard.ngrok.com/auth
MODEL       = "Qwen/Qwen2.5-VL-7B-Instruct"
PORT        = 8000

In [ ]:
# Cell 3 — Verify A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 4 — Start vLLM server in the background
import subprocess, time, sys, urllib.request

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--max-model-len", "8192",
    "--dtype", "bfloat16",
    "--limit-mm-per-prompt", '{"image": 5}',
]

log = open("/tmp/vllm.log", "w")
server_proc = subprocess.Popen(cmd, stdout=log, stderr=log)

print("Starting vLLM server — downloading model on first run (~10 min)...")
print("Waiting for server to be ready...")

for i in range(60):
    time.sleep(10)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health")
        print(f"\nServer ready after {(i+1)*10}s")
        break
    except Exception:
        print(f"  [{(i+1)*10}s] still loading...", end="\r")
else:
    print("\nTimed out — check /tmp/vllm.log for errors")

In [ ]:
# Cell 5 — Start ngrok tunnel and print config
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print("\n" + "=" * 60)
print("Paste this block into your backend .env then restart uvicorn:")
print("=" * 60)
print(f"OPENAI_BASE_URL={public_url}/v1")
print(f"OPENAI_API_KEY=no-key")
print(f"LLM_MODEL={MODEL}")
print("=" * 60)
print(f"\nAPI docs: {public_url}/docs")

In [ ]:
# Cell 6 — Quick smoke test (text only)
from openai import OpenAI

client = OpenAI(base_url=f"{public_url}/v1", api_key="no-key")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an expert agronomist."},
        {"role": "user",   "content": "Tomato leaves are curling inward and showing bronze speckling. What could it be?"}
    ],
    max_tokens=200,
)
print(response.choices[0].message.content)

In [ ]:
# Cell 7 — Keep-alive (run this to prevent Colab from disconnecting)
# Leave this cell running while you use the app.
import time
print("Keep-alive running. Stop this cell to shut down.")
while True:
    time.sleep(30)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health")
    except Exception:
        print("Server died — re-run Cell 4")
        break